## Topic 6: Other Types of Splitters & Examples

### Concept, Intuition, Why It Exists

- The previous topics covered the core, general-purpose splitters. Real ingestion pipelines eventually run into source content that doesn't fit prose, Q&A, or Markdown assumptions cleanly — code files, HTML pages, JSON-as-data (not JSON-as-page-export like this project already uses), and very long documents where even recursive character splitting still produces topically-mixed chunks. This topic rounds out the picture with the remaining splitter types worth knowing by name.
- **CodeTextSplitter**: a language-aware variant of recursive splitting where the separator priority list is built from a programming language's actual syntax (function/class boundaries, then blocks, then lines) instead of paragraph/line/space. The same recursive-fallback mechanics from the previous topic, just with separators chosen to respect code structure instead of prose structure.
- **HTMLHeaderTextSplitter**: the HTML sibling of the Markdown header splitter — splits on `<h1>`/`<h2>`/`<h3>` tags (or other structural tags) instead of `#`/`##`/`###`, attaching the same kind of header-hierarchy metadata to each chunk.
- **JSONSplitter (structured-data splitter)**: rather than treating JSON as text to split on characters, this walks the JSON's actual nested structure and emits one chunk per logical node/object, keeping each chunk valid and self-contained relative to its place in the structure — directly relevant to this project's own JSON loader if a future JSON source has deeply nested data rather than the current flat `pages` list.
- **Agentic/LLM-based chunking**: instead of any rule-based separator logic at all, an LLM itself is prompted to read a document and propose chunk boundaries based on its own understanding of where ideas naturally divide. The most expensive option by far — every document costs an LLM call just to be chunked — but can outperform every rule-based approach on genuinely irregular content where no separator-based or embedding-similarity heuristic captures the actual structure.

### Internal Working

- **CodeTextSplitter**: recursively splits using a separator list ordered specifically for the target language — e.g. for Python: class definition, then function definition, then logical block (if/for/while), then line, then character — falling back exactly the same way Topic 5's recursive splitter does, just with code-aware separators instead of prose-aware ones.
- **HTMLHeaderTextSplitter**: parses the HTML tree, identifies header tags at each level, and treats the content between one header and the next (within the same nesting level) as one structural unit — mechanically identical to the Markdown header splitter from Topic 4, operating on a different markup syntax.
- **JSONSplitter**: recursively walks the JSON object graph; for each key/value pair or array item, decides whether the serialized sub-structure fits within the size budget as one chunk, and if not, recurses one level deeper into that sub-structure — the same recursive divide-until-it-fits logic as Topic 5's character splitter, but operating on structural nesting instead of separator characters.
- **Agentic chunking**: send the full document (or large windows of it) to an LLM with a prompt asking it to return chunk boundaries (or pre-split text) based on topical coherence; parse the LLM's response back into discrete chunks. No separator logic at all — the LLM's own judgment replaces every rule used by every other splitter in this sub-chapter.

### How It's Implemented In This Project

- None of these are currently needed by this project's actual source types (text references, CSV/Excel rows, JSON page exports) — they're documented here because they're the natural next splitters to reach for the moment new source types are added: ingesting actual source code documentation would call for CodeTextSplitter; ingesting a scraped HTML knowledge base would call for HTMLHeaderTextSplitter; a future JSON source with deeply nested (not flat) structure would call for a JSON-structure-aware splitter instead of treating that JSON as flat text.
- Agentic chunking is explicitly **not** recommended as this project's default given its cost — flagged here mainly so it's recognized by name and reserved for genuinely irregular content where every cheaper option has been tried and measurably underperforms.

### Real-World Issues, Edge Cases, Debugging

- **CodeTextSplitter needs the correct language grammar to be useful at all** — using Python-tuned separators on a JavaScript file produces boundaries that don't actually align with that language's real structure, the same mismatch risk as using a mismatched tokenizer in Topic 4's TokenTextSplitter.
- **HTMLHeaderTextSplitter inherits the web's structural inconsistency** — real-world HTML is frequently malformed or uses headers purely for visual styling rather than genuine document structure (a `<h3>` used just because it "looks right" in a particular spot, with no real hierarchical meaning) — the same structure-reliability risk flagged for Markdown headers in Topic 4, just worse in practice because HTML in the wild is messier than authored Markdown.
- **JSONSplitter's chunk size vs. structural integrity trade-off**: forcing a strict size limit onto JSON can produce a chunk that's a syntactically valid but semantically incomplete fragment of a larger object (half of an array, with no indication the rest exists elsewhere) — worth deciding explicitly whether to allow oversized-but-complete chunks for some structures rather than always enforcing the size limit strictly.
- **Agentic chunking's biggest production risk is non-determinism and cost**: re-running the same document through an LLM-based chunker can produce slightly different boundaries each time, complicating the incremental-ingestion manifest pattern from the earlier sub-chapter (which assumes "same content in → same result out"), and the per-document LLM cost scales linearly with corpus size in a way none of the rule-based splitters do.

### Design Decisions & Trade-offs

- Rule-based splitters (everything except agentic chunking) trade some quality ceiling for predictability, speed, and cost — they're deterministic, fast, and free beyond the embedding cost already being paid anyway.
- Agentic chunking trades predictability and cost for a potentially higher quality ceiling on genuinely irregular content — only worth it when measured evidence shows rule-based splitters are failing on a specific content type, not adopted as a general-purpose default given its cost and non-determinism.
- Structure-aware splitters (Code, HTML, JSON) all share the same underlying trade-off already established for Markdown/Q&A splitters in earlier topics: highest quality when the assumed structure genuinely holds, brittle and silently degraded when it doesn't.

### Alternatives & When To Use Each

- CodeTextSplitter — ingesting source code or code-heavy technical documentation where function/class boundaries are the meaningful unit.
- HTMLHeaderTextSplitter — ingesting scraped or exported HTML content with genuine (not purely cosmetic) header structure.
- JSONSplitter — ingesting deeply nested structured JSON where flat text splitting would either break valid JSON syntax or ignore meaningful nesting.
- Agentic/LLM-based chunking — small volumes of genuinely irregular content where every cheaper, rule-based option has been tried and measurably underperforms; not a default for any high-volume pipeline.
- Recursive character splitting (Topic 5) — still the right general-purpose fallback for any content that doesn't cleanly fit one of the more specialized structures above.

### Common Mistakes & Production Failures

- Reaching for agentic chunking by default because it sounds like the most "advanced" option, without first confirming cheaper rule-based splitters are actually insufficient for the content.
- Using a structure-aware splitter (Code/HTML/JSON) on content where that structure is present but unreliable, and not validating chunk completeness afterward the same way every other structure-aware splitter in this sub-chapter has needed validation.
- Forgetting that non-deterministic chunking (agentic) breaks the "same input, same output" assumption that incremental ingestion's manifest pattern depends on — a model producing slightly different chunk boundaries on a re-run looks identical to the manifest (same file hash) but produces a different embedded result, a subtle mismatch worth being aware of before adopting it.

### Lead-Level Interview Questions

**Q: When would you justify the cost of agentic/LLM-based chunking over every rule-based alternative covered so far?**
A: Only after rule-based options (recursive character splitting, structure-aware splitting where applicable, semantic chunking) have been tried and shown, with actual evidence, to underperform on a specific content type — typically genuinely irregular content with no consistent separators, headers, or stable internal structure at all. It should never be a default; the cost (an LLM call per document) and non-determinism are real, ongoing costs, not one-time setup costs.

**Q: Why might using a structure-aware splitter on real-world scraped HTML be riskier than using one on hand-authored Markdown?**
A: Markdown headers are almost always used semantically — when someone writes `##`, they mean "this is a real subsection." HTML headers in scraped, real-world content are frequently used for visual styling rather than genuine document structure, so the same structure-aware splitting logic inherits a less reliable signal, producing structural metadata (and chunk boundaries) that look meaningful but may not actually reflect the document's real logical organization.

**Q: Agentic chunking re-run on the same document produces slightly different chunk boundaries each time. What does this break elsewhere in the pipeline?**
A: It breaks the implicit assumption behind the incremental ingestion manifest from the earlier sub-chapter — that manifest decides "skip re-ingestion" purely based on whether the source file's content hash changed, assuming the same input always deterministically produces the same chunked output. With non-deterministic chunking, a file with an unchanged hash could still, if reprocessed for any reason, produce a different embedded result than what's already in the vector store, an inconsistency rule-based splitters don't introduce.

### Hidden Concepts Worth Knowing

- **Determinism as a cross-cutting pipeline requirement**: every rule-based splitter in this sub-chapter is deterministic (same input always produces the same chunks), which is what makes the incremental-ingestion manifest pattern, reproducible debugging, and consistent retrieval behavior all work correctly. Introducing any non-deterministic stage (agentic chunking, or any other LLM-in-the-loop ingestion step) is a decision with consequences well beyond chunk quality alone.
- **The full splitter landscape is really one recurring idea, applied to different structures**: prose → recursive character splitting; Q&A/Markdown/HTML → structure markers; code → language grammar; nested data → object graph traversal; irregular content → LLM judgment as a last resort. Every splitter covered across this entire sub-chapter is the same underlying question — "what is the most reliable signal available in this content for where a complete idea ends?" — answered differently per content type.

### Revision Summary

> Beyond the core general-purpose splitters, specialized variants exist for code (language-grammar-aware separators), HTML (header-tag structure), and nested JSON (object-graph traversal), each inheriting the same structure-reliability trade-off as Markdown/Q&A splitters. Agentic/LLM-based chunking removes rule-based logic entirely in favor of model judgment, at real cost and non-determinism — reserved for irregular content where every cheaper option has measurably failed, and notably incompatible with the determinism the incremental-ingestion manifest pattern assumes.

In [6]:
"""
other_splitters.py
---------------------
Worked examples of three further splitter types: a code-aware recursive
splitter, an HTML header splitter, and a structure-aware JSON splitter.
Agentic/LLM-based chunking is described in theory only -- no code example,
since it's just "send text to an LLM with a chunking prompt and parse the
response," with no reusable mechanics beyond a prompt template.
"""

import re
import json
from document import make_document
from recursive_text_splitter import recursive_character_split  # reuse Topic 5's core logic

# ----------------------------------------------------------------------
# 1. CodeTextSplitter -- recursive splitting with language-aware separators
# ----------------------------------------------------------------------
PYTHON_SEPARATORS = [
    "\nclass ",      # class boundaries first -- highest-level structural unit
    "\ndef ",        # then function boundaries
    "\n\n",          # then blank-line-separated blocks
    "\n",            # then individual lines
    " ",              # then words
    "",               # absolute last resort: character cut
]


def code_text_split(code: str, chunk_size: int = 200, separators: list = None) -> list:
    """Same recursive mechanics as Topic 5, just with code-aware separators."""
    return recursive_character_split(code, chunk_size=chunk_size, separators=separators or PYTHON_SEPARATORS)


# ----------------------------------------------------------------------
# 2. HTMLHeaderTextSplitter -- splits on <h1>/<h2>/<h3>, mirrors the
#    Markdown header splitter from Topic 4 but for HTML tags.
# ----------------------------------------------------------------------
HTML_HEADER_RE = re.compile(r"<h([1-3])>(.*?)</h\1>", re.IGNORECASE)


def html_header_split(html: str) -> list:
    matches = list(HTML_HEADER_RE.finditer(html))
    if not matches:
        return [make_document(html.strip(), {"header_path": []})]

    documents = []
    header_stack = []

    for i, match in enumerate(matches):
        level = int(match.group(1))
        title = match.group(2).strip()

        header_stack = [h for h in header_stack if h[0] < level]
        header_stack.append((level, title))

        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(html)
        section_text = re.sub(r"<[^>]+>", "", html[start:end]).strip()  # strip remaining tags

        if section_text:
            documents.append(make_document(
                page_content=section_text,
                metadata={"header_path": [h[1] for h in header_stack]},
            ))

    return documents


# ----------------------------------------------------------------------
# 3. JSONSplitter -- structure-aware: walks the object graph, recursing
#    only into sub-structures that are still too large.
# ----------------------------------------------------------------------
def json_structure_split(data, chunk_size: int = 200, path: str = "$") -> list:
    serialized = json.dumps(data, ensure_ascii=False)

    if len(serialized) <= chunk_size:
        return [make_document(page_content=serialized, metadata={"json_path": path})]

    documents = []
    if isinstance(data, dict):
        for key, value in data.items():
            documents.extend(json_structure_split(value, chunk_size, path=f"{path}.{key}"))
    elif isinstance(data, list):
        for i, item in enumerate(data):
            documents.extend(json_structure_split(item, chunk_size, path=f"{path}[{i}]"))
    else:
        # a single scalar value that's somehow still oversized (e.g. a huge string)
        documents.append(make_document(page_content=serialized, metadata={"json_path": path}))

    return documents


if __name__ == "__main__":
    code_sample = (
        "class FDValidator:\n"
        "    def validate(self, fd_no):\n"
        "        return fd_no.startswith('BJ')\n\n"
        "    def normalize(self, fd_no):\n"
        "        return fd_no.strip().upper()\n"
    )
    print("--- CodeTextSplitter ---")
    for c in code_text_split(code_sample, chunk_size=60):
        print(f"  {c!r}")

    html_sample = (
        "<h1>FD Policy</h1><p>Overview text.</p>"
        "<h2>Premature Withdrawal</h2><p>A 1 percent penalty applies.</p>"
        "<h2>Senior Citizen Rate</h2><p>An additional 0.5 percent applies.</p>"
    )
    print("\n--- HTMLHeaderTextSplitter ---")
    for d in html_header_split(html_sample):
        print(f"  header_path={d['metadata']['header_path']}: {d['page_content']!r}")

    json_sample = {
        "document_code": "FD-PROD-02",
        "pages": [
            {"page_number": 1, "text": "Short page text."},
            {"page_number": 2, "text": "A much longer page of text that would exceed a small chunk size limit on its own and needs to be considered separately from page 1."},
        ],
    }
    print("\n--- JSONSplitter ---")
    for d in json_structure_split(json_sample, chunk_size=80):
        print(f"  json_path={d['metadata']['json_path']}: {d['page_content'][:60]!r}")

--- CodeTextSplitter ---
  'class FDValidator:\nclass     def validate(self, fd_no):'
  "        return fd_no.startswith('BJ')"
  '    def normalize(self, fd_no):'
  '        return fd_no.strip().upper()\nclass '

--- HTMLHeaderTextSplitter ---
  header_path=['FD Policy']: 'Overview text.'
  header_path=['FD Policy', 'Premature Withdrawal']: 'A 1 percent penalty applies.'
  header_path=['FD Policy', 'Senior Citizen Rate']: 'An additional 0.5 percent applies.'

--- JSONSplitter ---
  json_path=$.document_code: '"FD-PROD-02"'
  json_path=$.pages[0]: '{"page_number": 1, "text": "Short page text."}'
  json_path=$.pages[1].page_number: '2'
  json_path=$.pages[1].text: '"A much longer page of text that would exceed a small chunk '
